# Explicit PDB connectivity profiles and native candidates

Evidence for DockingMT #33 and MolSysMT #304. The current default reader uses
optional OpenMM inference; its absence leaves an incomplete explicit-only graph.
The residue audit correctly reports that graph. This notebook reproduces both
profiles and compares existing native candidates without applying any bonds.
See `pdb_bond_inference.md` for contracts, axes, exclusions and the CI correction.

Run in `molsyssuite@uibcdf_3.14` with exact provider source
`e8e4fff22d0df0d26a3b91d80ea5a85c04981aef` on `PYTHONPATH`. OpenMM is required
for the comparison profile, not the candidate inference calls. This notebook
writes its own raw record and preserves previous preparation benchmarks.

In [1]:
import json
from pathlib import Path
import runpy
import sys
import pandas as pd

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'devtools/probe_pdb_bond_inference.py').is_file())
namespace = runpy.run_path(str(root / 'devtools/probe_pdb_bond_inference.py'))
print('Interpreter:', sys.executable)
namespace['main']()
payload = json.loads((root / 'devguide/validation/data/pdb_bond_inference/probe.json').read_text())
print(json.dumps({key: value for key, value in payload.items() if key != 'cases'}, indent=2))

Interpreter: /home/diego/Myopt/miniconda3/envs/molsyssuite@uibcdf_3.14/bin/python


/home/diego/repos@uibcdf/dockingmt/devguide/validation/data/pdb_bond_inference/probe.json
181L explicit: 13 OpenMM: 1322 native-only: 0 OpenMM-only: 0
1IEP explicit: 0 OpenMM: 4469 native-only: 2 OpenMM-only: 16
{
  "evidence": "Read-only source-specific candidate comparison, not independent chemical validation or a performance benchmark",
  "provider_reference_source": "e8e4fff22d0df0d26a3b91d80ea5a85c04981aef",
  "provider_import": "/tmp/dockingmt-receptor-provider-e8e4fff22d0df0d26a3b91d80ea5a85c04981aef/molsysmt/__init__.py",
  "python": "3.14.7",
  "interpreter": "/home/diego/Myopt/miniconda3/envs/molsyssuite@uibcdf_3.14/bin/python",
  "versions": {
    "openmm": "8.6.1",
    "numpy": "2.4.6",
    "pandas": "2.3.3"
  },
  "implementation_sha256": {
    "build/get_missing_bonds.py": "0e198912aacf8ce39ab8e52ca2a38eade037e3a1103e0ebd5844aee727b85416",
    "form/file_pdb/to_molsysmt_MolSys.py": "4985021a838a6f99893d3ac6eea0c5a864b6ee64b2f1ce32338d4f9a3f4747d2",
    "form/molsysmt_PDBF

## Actual graph and coverage depend on the reader policy

With OpenMM imports blocked, the current file reader retains only explicitly
recorded edges. The public handler with `get_missing_bonds=False` yields the
same graph. At this provider source the direct file conversion silently discards
that flag; its correction is requested upstream. Heavy-atom presence remains
separate from complete stored connectivity. No input bytes are rewritten.

In [2]:
display(pd.DataFrame([{
    'case': case['case'], 'source_atoms': case['source_atoms'],
    'explicit_handler_edges': case['explicit_handler_bonds'],
    'without_OpenMM_edges': case['without_openmm_bonds'],
    'OpenMM_edges': case['openmm_bonds'],
    'file_flag_false_edges': case['file_get_missing_bonds_false_bonds'],
    **case['without_openmm_summary'],
} for case in payload['cases']]))
print(json.dumps([{'case': case['case'], 'reasons': case['without_openmm_reason_counts']}
                  for case in payload['cases']], indent=2))

,case,source_atoms,explicit_handler_edges,without_OpenMM_edges,OpenMM_edges,file_flag_false_edges,assessed,incomplete,unassessed
0,181L,1441,13,13,1322,1322,0,162,140
1,1IEP,4412,0,0,4469,4469,0,274,0


[
  {
    "case": "181L",
    "reasons": {
      "incomplete_stored_connectivity": 162,
      "no_exact_residue_template": 140
    }
  },
  {
    "case": "1IEP",
    "reasons": {
      "incomplete_stored_connectivity": 274
    }
  }
]


## A reusable native foundation, not yet a qualified replacement

`msm.build.get_missing_bonds(engine='MolSysMT', pbc=False)` runs with OpenMM
imports blocked. Reference-selected source atom indices are reused on the
explicit-only input, verifying names, elements and IDs before comparing edge
sets. Connectivity itself can change molecule membership, so separately
selecting `molecule_type=='protein'` on each graph would change the population.

Pairs below use the extracted selection's atom axis; the raw record retains its
complete source-index map. Native candidates remain unapplied. The 181L edge
sets agree exactly. The 1IEP differences and stdout warnings require investigation;
agreement with OpenMM would still not supply independent chemical validation,
bond orders, protonation, charge models, force-field types or docking readiness.
No timing or memory claim is made.

In [3]:
display(pd.DataFrame([{
    'case': case['case'], 'selected_atoms': case['selected_atoms'],
    'native_candidate_edges': case['native_candidate_bonds'],
    'OpenMM_comparison_edges': case['openmm_comparison_bonds'],
    'native_only_edges': len(case['native_only_pairs']),
    'OpenMM_only_edges': len(case['openmm_only_pairs']),
    'native_stdout_lines': case['native_stdout_lines'],
} for case in payload['cases']]))
for case in payload['cases']:
    print(case['case'], 'native-only pairs:', case['native_only_pairs'],
          'OpenMM-only pairs:', case['openmm_only_pairs'])
print('Full source maps, implementation hashes and native stdout are retained in probe.json.')

,case,selected_atoms,native_candidate_edges,OpenMM_comparison_edges,native_only_edges,OpenMM_only_edges,native_stdout_lines
0,181L,1289,1309,1309,0,0,0
1,1IEP,4412,4455,4469,2,16,224


181L native-only pairs: [] OpenMM-only pairs: []
1IEP native-only pairs: [[0, 3427], [4398, 4399]] OpenMM-only pairs: [[252, 254], [297, 299], [684, 686], [762, 764], [844, 846], [1312, 1314], [1472, 1474], [1538, 1540], [1958, 1960], [2666, 2668], [2699, 2701], [2734, 2736], [2924, 2926], [3368, 3370], [3408, 3421], [4349, 4351]]
Full source maps, implementation hashes and native stdout are retained in probe.json.
